In [33]:
print("ok")

ok


In [34]:
from langchain.agents import create_agent
from langchain.agents import AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.tools import tool, ToolRuntime



In [35]:
from dotenv import load_dotenv
load_dotenv()
import os
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [36]:

from langchain_groq import ChatGroq
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,)

In [37]:
client1 = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "/opt/anaconda3/envs/student-helper/bin/python",
            "args": [
                "/Users/oluwaferanmi/Documents/Sibyl/mcp_server.py"
            ],
        }
    }
)

In [38]:
pdf_parsing_tools = await client1.get_tools()

In [39]:
from dataclasses import dataclass
@dataclass
class CustomState(AgentState):
    username: str
    academic_standing: str
    university: str

In [40]:
prompt="""
You are SIBYL, an advanced Academic Strategist Agent. 
Your goal is not just to "put things on a calendar," but to optimize the student's semester for maximum GPA with minimum stress.

### YOUR CAPABILITIES
1. **Scan Syllabi:** You can read entire folders of PDF syllabi using `scan_syllabus_folder`.
2. **Execute Strategy:** You can write directly to the user's Google Calendar using `add_calendar_event`.

### RULES OF ENGAGEMENT
1. **Always Verify First:** - Never schedule events immediately after scanning. 
   - First, present a summary: "I found 12 events. 3 Exams (High Priority), 4 Papers, and 5 Readings."
   - Ask: "Would you like me to schedule the High Priority items first, or everything?"

2. **The "Strategy" Logic:**
   - When the user says "Go ahead," use the `strategy` fields provided by the scanner tool.
   - For Exams: Schedule the "Study Start Date" (Red/High priority).
   - For Papers: Schedule the "Drafting Start Date" (Yellow/Medium priority).
   - **Crucial:** Always include the original "Due Date" in the calendar description so the user knows when the hard deadline is.
   
3. **Handling Multiple Courses:**
   - You will often receive multiple courses at once. 
   - You must generate separate calendar tool calls for every single event in every single course. 
   - If there are >5 events total, ask for confirmation before firing all API calls to avoid overwhelming the user.
   - If there are more than 5 events, process them in batches or ask for explicit permission to do all at once to avoid API rate limits.

4. **Tone & Persona:**
   - You are cool, precise, and slightly futuristic (anime-inspired AI). 
   - Keep responses concise. Do not lecture.
   - If an error occurs (like a missing file), be specific about what went wrong.

### HANDLING "ADD TO CALENDAR" REQUESTS
When calling the `add_calendar_event` tool:
- Ensure 'start_date' is strictly YYYY-MM-DD.
- Use the 'priority' argument: 'high' for Exams, 'medium' for Assignments, 'low' for Readings.
"""

In [41]:

from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_username(username: str, runtime: ToolRuntime) -> Command:
    """ Update the username of the user in the state once they've revealed it"""
    return Command[tuple[()]](update={
        "username": username,
        "messages": [ToolMessage("Successfully updated user's username", tool_call_id=runtime.tool_call_id)]
    })
@tool
def update_user_university(university: str, runtime: ToolRuntime) -> Command:
    """ Update the university of the user in the state once they've revealed it"""
    return Command[tuple[()]](update={
        "university": university,
        "messages": [ToolMessage("Successfully updated user's university", tool_call_id=runtime.tool_call_id)]
    })
@tool
def update_user_academic_standing(academic_standing : str, runtime: ToolRuntime) -> Command:
    """ Update the academic standing of the user(like freshman, sophmore) in the state once they've revealed it"""
    return Command[tuple[()]](update={
        "academic_standing": academic_standing,
        "messages": [ToolMessage("Successfully updated user's academic standing", tool_call_id=runtime.tool_call_id)]
    })

In [42]:
agent= create_agent(
    model=model, 
    tools=[update_user_academic_standing, update_user_university,update_username, *pdf_parsing_tools],
    system_prompt=prompt,
    checkpointer=InMemorySaver(),
    state_schema= CustomState
    )

In [43]:
config={"configurable": {"thread_id":"1"}}
from langchain.messages import HumanMessage
question=HumanMessage(content="Hello. I am Oluwaferanmi Oyelude.")
response=await agent.ainvoke({"messages": [question]}, config)

In [44]:
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='Hello. I am Oluwaferanmi Oyelude.', additional_kwargs={}, response_metadata={}, id='5196a215-3f50-4744-bc6f-818a8401825c'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '58tkvhnp7', 'function': {'arguments': '{"username":"Oluwaferanmi Oyelude"}', 'name': 'update_username'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 1257, 'total_tokens': 1280, 'completion_time': 0.048994882, 'completion_tokens_details': None, 'prompt_time': 0.06866355, 'prompt_tokens_details': None, 'queue_time': 0.008509768, 'total_time': 0.117658432}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b626e-c477-76b3-b517-855a683fd664-0', tool_calls=[{'name': 'update_username', 'args': {'username': 'Oluwaferanmi Oyelude'}, 'id': '58tkvhnp7', 'type'

In [45]:
question=HumanMessage(content="My university is Howard University and i am a freshamn")
response1=await agent.ainvoke({"messages": [question]}, config)

In [46]:
from pprint import pprint
pprint(response1)

{'academic_standing': 'freshman',
 'messages': [HumanMessage(content='Hello. I am Oluwaferanmi Oyelude.', additional_kwargs={}, response_metadata={}, id='5196a215-3f50-4744-bc6f-818a8401825c'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '58tkvhnp7', 'function': {'arguments': '{"username":"Oluwaferanmi Oyelude"}', 'name': 'update_username'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 1257, 'total_tokens': 1280, 'completion_time': 0.048994882, 'completion_tokens_details': None, 'prompt_time': 0.06866355, 'prompt_tokens_details': None, 'queue_time': 0.008509768, 'total_time': 0.117658432}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b626e-c477-76b3-b517-855a683fd664-0', tool_calls=[{'name': 'update_username', 'args': {'username': 'Oluwaferanmi Oy

In [47]:
question=HumanMessage(content="I want you to scan my syllabi. This is the path to it: /Users/oluwaferanmi/Documents/syllabus_folder")
response1=await agent.ainvoke({"messages": [question]}, config)

In [60]:
pprint(response1)

{'academic_standing': 'freshman',
 'messages': [HumanMessage(content='Hello. I am Oluwaferanmi Oyelude.', additional_kwargs={}, response_metadata={}, id='5196a215-3f50-4744-bc6f-818a8401825c'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '58tkvhnp7', 'function': {'arguments': '{"username":"Oluwaferanmi Oyelude"}', 'name': 'update_username'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 1257, 'total_tokens': 1280, 'completion_time': 0.048994882, 'completion_tokens_details': None, 'prompt_time': 0.06866355, 'prompt_tokens_details': None, 'queue_time': 0.008509768, 'total_time': 0.117658432}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b626e-c477-76b3-b517-855a683fd664-0', tool_calls=[{'name': 'update_username', 'args': {'username': 'Oluwaferanmi Oy

In [52]:
pprint(response1["messages"][-2].content)

[{'id': 'lc_34f059ac-6ae0-44cf-bc7f-6d7159eb703f',
  'text': '{\n'
          '  "page_content": "ENGW 104– Writing, Literacy, and Discourse '
          '\\nFall 2025 \\nCourse Instructor: Prof. Kenyatta Graves    '
          '\\nClass Time: Section 27—MWF 10:10a-11:00a; Section 28—MWF '
          '11:10a-12:00p    \\nEmail Address: kenyatta.graves@howard.edu   '
          '\\nOffice Hours:  Mondays and Wednesday—2:00pm to 5:00pm; Tuesdays '
          '4:00pm-7:00pm. All \\noffice hours are virtual, via Zoom and by '
          'appointment only. Additional days/hours available by \\nrequest via '
          'email. \\nWriting For Your Life: Identity, Space, and Place '
          '\\n“Word-work is sublime, she thinks, because it is generative; it '
          '\\nmakes meaning that secures our difference, our human '
          '\\ndifference – the way in which we are like no other life. \\nWe '
          'die. That may be the meaning of life. But we do language. \\nThat '
          'may be

In [ ]:
question=HumanMessage(content="Load everything")
response=await agent.ainvoke({"messages": [question]}, config)

Process group termination failed for PID 37002: [Errno 1] Operation not permitted, falling back to simple terminate
Process group termination failed for PID 37006: [Errno 1] Operation not permitted, falling back to simple terminate
Process group termination failed for PID 37010: [Errno 1] Operation not permitted, falling back to simple terminate
Process group termination failed for PID 37027: [Errno 1] Operation not permitted, falling back to simple terminate
Process group termination failed for PID 36996: [Errno 1] Operation not permitted, falling back to simple terminate


In [51]:
pprint(response)

{'academic_standing': 'freshman',
 'messages': [HumanMessage(content='Hello. I am Oluwaferanmi Oyelude.', additional_kwargs={}, response_metadata={}, id='5196a215-3f50-4744-bc6f-818a8401825c'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '58tkvhnp7', 'function': {'arguments': '{"username":"Oluwaferanmi Oyelude"}', 'name': 'update_username'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 1257, 'total_tokens': 1280, 'completion_time': 0.048994882, 'completion_tokens_details': None, 'prompt_time': 0.06866355, 'prompt_tokens_details': None, 'queue_time': 0.008509768, 'total_time': 0.117658432}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b626e-c477-76b3-b517-855a683fd664-0', tool_calls=[{'name': 'update_username', 'args': {'username': 'Oluwaferanmi Oy

In [53]:
question=HumanMessage(content="Can you list all the events you added?")
response=await agent.ainvoke({"messages": [question]}, config)

In [ ]:
pprint(response)

{'academic_standing': 'freshman',
 'messages': [HumanMessage(content='Hello. I am Oluwaferanmi Oyelude.', additional_kwargs={}, response_metadata={}, id='5196a215-3f50-4744-bc6f-818a8401825c'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '58tkvhnp7', 'function': {'arguments': '{"username":"Oluwaferanmi Oyelude"}', 'name': 'update_username'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 1257, 'total_tokens': 1280, 'completion_time': 0.048994882, 'completion_tokens_details': None, 'prompt_time': 0.06866355, 'prompt_tokens_details': None, 'queue_time': 0.008509768, 'total_time': 0.117658432}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019b626e-c477-76b3-b517-855a683fd664-0', tool_calls=[{'name': 'update_username', 'args': {'username': 'Oluwaferanmi Oy

In [59]:
question=HumanMessage(content="Thank you")
response=await agent.ainvoke({"messages": [question]}, config)

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.3-70b-versatile` in organization `org_01kbd6xfyvfz182q6aegkhncd7` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Requested 12529, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}